# P-ML6 — LSTM Forecaster

**Motivation (F12):** P-ML5 `RegimeEnsemble` achieves Sharpe +0.927 on a 6yr OOS window,
but still trails Buy & Hold (+1.379). All prior experiments (P-ML2 through P-ML5) use
tree-based LightGBM, which receives a **single bar's features** as input — it cannot
exploit multi-bar temporal patterns in the feature sequence.

**Hypothesis for P-ML6:** An LSTM can capture **sequential dependencies across 30 bars**
(~1 month daily) that a single-bar LightGBM cannot, improving OOS IC and equity.

**Architecture:** Pure sequence model (no regime gating). Same 6yr dataset (2019–2025),
same 12-feature set, same purged walk-forward (5 folds, train_frac=0.6, purge=1).
Direct apples-to-apples comparison against P-ML5 LightGBM baseline.

## §1 — Config

In [1]:
import sys
from pathlib import Path

repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi":        120,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         9,
})

# ── Dataset ───────────────────────────────────────────────────────────────────
SYMBOL     = "BTC/USDT"
SINCE      = "2019-01-01"
UNTIL      = "2025-01-01"
HORIZON    = 1              # 1d forward log-return

# ── Walk-forward ──────────────────────────────────────────────────────────────
N_SPLITS   = 5
TRAIN_FRAC = 0.6
PURGE      = 1

# ── LSTM hyperparameters ──────────────────────────────────────────────────────
SEQ_LEN    = 30     # 30-bar lookback (~1 month daily)
LSTM_UNITS = 64
DROPOUT    = 0.2
EPOCHS     = 100    # EarlyStopping will terminate earlier
BATCH_SIZE = 32
LR         = 1e-3

# ── Features (same 12-feature set as P-ML2 through P-ML5) ────────────────────
FEATURES = [
    "bar_ret", "bb_zscore", "rsi", "macd_hist_norm", "atr_pct",
    "bb_width", "upper_wick", "lower_wick", "hl_range",
    "vol_log_chg", "di_diff", "adx",
]

# ── Static reference metrics from prior experiments ───────────────────────────
BUY_HOLD   = {"return": 2.996, "sharpe": 1.379, "maxdd": -0.354}
P_ML3_EXPC = {"return": 0.088, "sharpe": 0.280, "maxdd": -0.498}
P_ML5      = {"return": 6.302, "sharpe": 0.927, "maxdd": -0.680}

print(f"Dataset:       {SINCE} → {UNTIL} | 1d | horizon={HORIZON}")
print(f"Walk-forward:  {N_SPLITS} folds, train_frac={TRAIN_FRAC}, purge={PURGE}")
print(f"LSTM:          seq_len={SEQ_LEN}, units={LSTM_UNITS}, dropout={DROPOUT}")
print(f"               epochs={EPOCHS}, batch_size={BATCH_SIZE}, lr={LR}")
print(f"Features:      {len(FEATURES)} — {FEATURES}")

Dataset:       2019-01-01 → 2025-01-01 | 1d | horizon=1
Walk-forward:  5 folds, train_frac=0.6, purge=1
LSTM:          seq_len=30, units=64, dropout=0.2
               epochs=100, batch_size=32, lr=0.001
Features:      12 — ['bar_ret', 'bb_zscore', 'rsi', 'macd_hist_norm', 'atr_pct', 'bb_width', 'upper_wick', 'lower_wick', 'hl_range', 'vol_log_chg', 'di_diff', 'adx']


## §2 — Data Loading

Same 6yr fetch + cache-guard as P-ML5. Build feature matrix and forward-return label.

In [2]:
from data.fetch import fetch_ohlcv
from ml.features import build_feature_matrix
from ml.labels import forward_return
from ml.validation import purged_wf_splits

# Cache-guard: ensure we have 2019+ data
df_raw = fetch_ohlcv(symbol=SYMBOL, timeframe="1d", since=SINCE, until=UNTIL)
if df_raw.index.min() > pd.Timestamp("2020-01-01", tz="UTC"):
    print("Cache missing early data — re-fetching 2019–2025 from exchange...")
    df_raw = fetch_ohlcv(symbol=SYMBOL, timeframe="1d",
                         since=SINCE, until=UNTIL, use_cache=False)

print(f"Loaded {len(df_raw):,} bars  |  {df_raw.index[0].date()} → {df_raw.index[-1].date()}")

# Build features and labels
feats = build_feature_matrix(df_raw)
label = forward_return(df_raw, horizon=HORIZON)
comb  = pd.concat([feats, label], axis=1).dropna()

X_all = comb[feats.columns]
y_all = comb[label.name]
X     = X_all[FEATURES]

bar_ret_daily = np.log(df_raw["close"] / df_raw["close"].shift(1)).reindex(X.index)
splits = list(purged_wf_splits(len(X), N_SPLITS, TRAIN_FRAC, purge_bars=PURGE))

print(f"\n{len(X):,} usable bars | {X.index[0].date()} → {X.index[-1].date()}")
print(f"Features:  {X.shape[1]}  |  Label: {label.name}")
print(f"Folds:     {len(splits)} (after purge filter)")

Cache missing early data — re-fetching 2019–2025 from exchange...


Loaded 2,193 bars  |  2019-01-01 → 2025-01-01

2,171 usable bars | 2019-01-22 → 2024-12-31
Features:  12  |  Label: fwd_ret_1
Folds:     5 (after purge filter)


## §3 — Walk-Forward Validation with LSTMForecaster

Per fold: fit `LSTMForecaster(seq_len=30)`, predict OOS, compute Spearman IC and hit-rate.

**Warmup exclusion:** The first `seq_len - 1 = 29` predictions are zeros (prepended by
`LSTMForecaster.predict()`). These are excluded from IC and hit-rate computation.

In [3]:
from ml.models import LSTMForecaster

fold_results_P6 = []

print(f"{'Fold':<5} {'Period':<35} {'n_train':>8} {'n_test':>7} "
      f"{'epochs':>7} {'IC':>8} {'hit_rate':>9}")
print("-" * 82)

for i, (tr, te) in enumerate(splits):
    model = LSTMForecaster(
        seq_len=SEQ_LEN,
        units=LSTM_UNITS,
        dropout=DROPOUT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        verbose=0,
    )
    model.fit(X.iloc[tr], y_all.iloc[tr])
    preds = model.predict(X.iloc[te])   # length == len(te), first seq_len-1 are zeros

    # Exclude warmup bars from IC / hit-rate
    warmup = SEQ_LEN - 1
    if len(te) > warmup:
        valid_preds  = preds[warmup:]
        valid_actual = y_all.iloc[te].values[warmup:]
    else:
        valid_preds  = preds
        valid_actual = y_all.iloc[te].values

    rho, pval = stats.spearmanr(valid_preds, valid_actual)
    hit       = (np.sign(valid_preds) == np.sign(valid_actual)).mean()

    period = f"{X.index[te[0]].date()} → {X.index[te[-1]].date()}"
    print(f"  {i+1:<3} {period:<35} {len(tr):>8} {len(te):>7} "
          f"{model.epochs_stopped_:>7} {rho:>+8.4f} {hit:>9.3f}")

    fold_results_P6.append({
        "fold":           i + 1,
        "te":             te,
        "IC":             rho,
        "IC_pval":        pval,
        "hit":            hit,
        "preds":          preds,
        "epochs_stopped": model.epochs_stopped_,
        "model":          model,
    })

ics_P6 = [r["IC"] for r in fold_results_P6]
print()
print(f"P-ML6 LSTM: Mean IC={np.mean(ics_P6):+.4f}  ICIR={np.mean(ics_P6)/np.std(ics_P6):.3f}")

# Reference ICs from P-ML5 LightGBM (from F12)
p5_ics = [0.0612, -0.0536, 0.1295, 0.0462, 0.0861]
print(f"P-ML5 LightGBM: Mean IC={np.mean(p5_ics):+.4f}  ICIR={np.mean(p5_ics)/np.std(p5_ics):.3f}  [reference]")

Fold  Period                               n_train  n_test  epochs       IC  hit_rate
----------------------------------------------------------------------------------


2026-03-05 09:45:04.219422: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


  1   2020-01-18 → 2021-01-12                  360     361      23  -0.0451     0.464


  2   2021-01-13 → 2022-01-08                  540     361      23  +0.0223     0.524


  3   2022-01-09 → 2023-01-04                  540     361      24  +0.0011     0.485


  4   2023-01-05 → 2023-12-31                  540     361      24  -0.0061     0.518


  5   2024-01-01 → 2024-12-26                  540     361      15  +0.0271     0.503

P-ML6 LSTM: Mean IC=-0.0001  ICIR=-0.006
P-ML5 LightGBM: Mean IC=+0.0539  ICIR=0.888  [reference]


## §4 — Equity Curve: P-ML6 vs Baselines

Stitch OOS predictions → equity curve; compare against Buy & Hold, P-ML3 Exp-C, P-ML5.

In [4]:
from backtesting import compute_metrics


def build_equity(fold_preds_list, bar_ret_daily, X):
    """Stitch per-fold equity from a list of (te_idx, preds) tuples."""
    pieces, anchor = [], 1.0
    for te, preds in fold_preds_list:
        pos   = np.sign(preds)
        ret   = bar_ret_daily.iloc[te].values
        eq    = np.cumprod(1 + np.roll(pos, 1) * ret)
        eq[0] = 1.0
        s = pd.Series(eq, index=X.index[te])
        s = s / s.iloc[0] * anchor
        anchor = float(s.iloc[-1])
        pieces.append(s)
    return pd.concat(pieces)


# P-ML6 equity curve
p6_pairs = [(r["te"], r["preds"]) for r in fold_results_P6]
oos_P6   = build_equity(p6_pairs, bar_ret_daily, X)

# Buy-and-Hold anchored to OOS start
bah = df_raw["close"].reindex(oos_P6.index)
bah = bah / bah.iloc[0]

# Compute metrics
m_p6  = compute_metrics(oos_P6)
m_bah = compute_metrics(bah)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

oos_P6.plot(ax=ax, label="P-ML6 (LSTM)",      color="#e74c3c", linewidth=2.0)
bah.plot(   ax=ax, label="Buy-and-Hold",       color="black",   linewidth=1.0, linestyle=":")

for _, (tr, te) in enumerate(splits):
    ax.axvline(X.index[te[0]], color="lightgray", linewidth=0.5, linestyle="--")
ax.axhline(1, color="black", linewidth=0.4)
ax.set_title("OOS Equity — P-ML6 (LSTM) vs Buy-and-Hold", fontsize=11)
ax.set_ylabel("Equity (normalised)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ── Metrics table ─────────────────────────────────────────────────────────────
print(f"\n{'Strategy':<35} {'Return':>9} {'Sharpe':>8} {'MaxDD':>8}")
print("-" * 63)
rows = [
    ("Buy & Hold",                BUY_HOLD["return"],  BUY_HOLD["sharpe"],  BUY_HOLD["maxdd"]),
    ("P-ML3 Exp-C (best 3yr)",    P_ML3_EXPC["return"],P_ML3_EXPC["sharpe"],P_ML3_EXPC["maxdd"]),
    ("P-ML5 (6yr LightGBM)",      P_ML5["return"],     P_ML5["sharpe"],     P_ML5["maxdd"]),
    ("P-ML6 (LSTM)",               m_p6["total_return"], m_p6["sharpe_ratio"], m_p6["max_drawdown"]),
]
for name, ret, shr, mdd in rows:
    print(f"  {name:<33} {ret*100:>+8.1f}%  {shr:>+7.3f}  {mdd*100:>7.1f}%")


Strategy                               Return   Sharpe    MaxDD
---------------------------------------------------------------
  Buy & Hold                          +299.6%   +1.379    -35.4%
  P-ML3 Exp-C (best 3yr)                +8.8%   +0.280    -49.8%
  P-ML5 (6yr LightGBM)                +630.2%   +0.927    -68.0%
  P-ML6 (LSTM)                         -93.2%   -0.517    -94.7%


## §5 — LSTM vs LightGBM IC Deep-Dive

Fold-by-fold IC bar chart, training loss curve for a representative fold, and
consistency analysis (does LSTM reduce the number of negative-IC folds?).

In [5]:
# ── 5a: Fold-by-fold IC bar chart: P-ML5 LightGBM vs P-ML6 LSTM ───────────────
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(N_SPLITS)
w = 0.35

ax.bar(x - w/2, p5_ics,  w, label="P-ML5 LightGBM (6yr RegimeEnsemble)",
       color="#3498db", alpha=0.85)
ax.bar(x + w/2, ics_P6, w, label="P-ML6 LSTM (30-bar window)",
       color="#e74c3c", alpha=0.85)
ax.axhline(0,     color="black", linewidth=0.7)
ax.axhline(+0.03, color="gray",  linewidth=0.6, linestyle="--", label="±0.03 target")
ax.axhline(-0.03, color="gray",  linewidth=0.6, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels([f"F{i+1}" for i in range(N_SPLITS)])
ax.set_ylabel("OOS Spearman IC")
ax.set_title("OOS IC per fold: P-ML5 LightGBM vs P-ML6 LSTM", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ── 5b: Consistency summary ────────────────────────────────────────────────────
n_neg_p5 = sum(1 for ic in p5_ics  if ic < 0)
n_neg_p6 = sum(1 for ic in ics_P6  if ic < 0)
print(f"Negative-IC folds — P-ML5 LightGBM: {n_neg_p5}/{N_SPLITS}")
print(f"Negative-IC folds — P-ML6 LSTM:     {n_neg_p6}/{N_SPLITS}")
print()

print(f"{'Fold':<6} {'Period':<35} {'P-ML5 IC':>10} {'P-ML6 IC':>10} {'LSTM better?':>13}")
print("-" * 77)
for i, (r, p5_ic) in enumerate(zip(fold_results_P6, p5_ics)):
    _, te = splits[i]
    period = f"{X.index[te[0]].date()} → {X.index[te[-1]].date()}"
    better = "YES" if r["IC"] > p5_ic else "no"
    print(f"  {r['fold']:<4} {period:<35} {p5_ic:>+10.4f} {r['IC']:>+10.4f} {better:>13}")

n_better = sum(1 for r, p5 in zip(fold_results_P6, p5_ics) if r["IC"] > p5)
print(f"\nLSTM beats LightGBM in {n_better}/{N_SPLITS} folds")

Negative-IC folds — P-ML5 LightGBM: 1/5
Negative-IC folds — P-ML6 LSTM:     2/5

Fold   Period                                P-ML5 IC   P-ML6 IC  LSTM better?
-----------------------------------------------------------------------------
  1    2020-01-18 → 2021-01-12                +0.0612    -0.0451            no
  2    2021-01-13 → 2022-01-08                -0.0536    +0.0223           YES
  3    2022-01-09 → 2023-01-04                +0.1295    +0.0011            no
  4    2023-01-05 → 2023-12-31                +0.0462    -0.0061            no
  5    2024-01-01 → 2024-12-26                +0.0861    +0.0271            no

LSTM beats LightGBM in 1/5 folds


In [6]:
# ── 5c: Training loss curve for Fold 3 (representative mid-dataset fold) ───────
# Re-fit Fold 3 with verbose=1 to capture history — use a fresh model
rep_fold_idx = 2   # 0-based → Fold 3
tr3, te3 = splits[rep_fold_idx]

import tensorflow as tf
from tensorflow import keras
tf.get_logger().setLevel("ERROR")

from ml.features import build_feature_matrix
from sklearn.preprocessing import StandardScaler

# Build sequences manually to capture history
sc3    = StandardScaler()
X_sc3  = sc3.fit_transform(X.iloc[tr3][FEATURES].values)

seq_len = SEQ_LEN
n_win   = len(X_sc3) - seq_len + 1
n_feats = X_sc3.shape[1]
X_seq3  = np.empty((n_win, seq_len, n_feats), dtype=np.float32)
for i in range(n_win):
    X_seq3[i] = X_sc3[i : i + seq_len]
y_seq3 = y_all.iloc[tr3].values[seq_len - 1 :].astype(np.float32)

inp = keras.Input(shape=(seq_len, n_feats))
x   = keras.layers.LSTM(LSTM_UNITS, return_sequences=True)(inp)
x   = keras.layers.Dropout(DROPOUT)(x)
x   = keras.layers.LSTM(LSTM_UNITS // 2)(x)
x   = keras.layers.Dropout(DROPOUT)(x)
out = keras.layers.Dense(1)(x)

m3 = keras.Model(inp, out)
m3.compile(optimizer=keras.optimizers.Adam(learning_rate=LR), loss="mse")

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)
hist3 = m3.fit(
    X_seq3, y_seq3,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=0,
)

# Plot loss curve
fig, ax = plt.subplots(figsize=(10, 4))
ep = np.arange(1, len(hist3.history["loss"]) + 1)
ax.plot(ep, hist3.history["loss"],     label="train loss", color="steelblue", linewidth=1.5)
ax.plot(ep, hist3.history["val_loss"], label="val loss",   color="#e74c3c",   linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_title(f"Training loss curve — Fold 3 ({X.index[tr3[0]].date()} → {X.index[tr3[-1]].date()})",
             fontsize=10)
ax.legend(fontsize=9)
ax.axvline(len(ep), color="gray", linewidth=0.8, linestyle="--",
           label=f"stopped at epoch {len(ep)}")
plt.tight_layout()
plt.show()

print(f"Fold 3 training: {len(ep)} epochs (out of max {EPOCHS})")
print(f"  Final train loss: {hist3.history['loss'][-1]:.6f}")
print(f"  Final val loss:   {hist3.history['val_loss'][-1]:.6f}")

Fold 3 training: 22 epochs (out of max 100)
  Final train loss: 0.001907
  Final val loss:   0.001230


## §6 — Conclusions / Finding F13

Does the 30-bar LSTM improve on single-bar LightGBM at the 1-day horizon on 6yr BTC data?
Does temporal memory add signal beyond snapshot features?

In [7]:
p6_return = m_p6["total_return"]
p6_sharpe = m_p6["sharpe_ratio"]
p6_maxdd  = m_p6["max_drawdown"]

mean_ic_p6 = np.mean(ics_P6)
icir_p6    = mean_ic_p6 / np.std(ics_P6) if np.std(ics_P6) > 0 else float("nan")
mean_ic_p5 = np.mean(p5_ics)
icir_p5    = mean_ic_p5 / np.std(p5_ics)

lstm_beats_lgbm_ic     = mean_ic_p6 > mean_ic_p5
lstm_beats_lgbm_sharpe = p6_sharpe  > P_ML5["sharpe"]
lstm_beats_bah_sharpe  = p6_sharpe  > BUY_HOLD["sharpe"]

print("=" * 70)
print("FINDING F13 — LSTM Forecaster (P-ML6) vs LightGBM (P-ML5)")
print("=" * 70)
print()
print("Hypothesis: 30-bar LSTM captures multi-bar temporal dependencies that")
print("single-bar LightGBM cannot, improving OOS IC and equity on 1d BTC.")
print()
print(f"{'Metric':<30} {'P-ML5 LightGBM':>16} {'P-ML6 LSTM':>13}")
print("-" * 62)
print(f"  {'Mean OOS IC':<28} {mean_ic_p5:>+16.4f} {mean_ic_p6:>+13.4f}")
print(f"  {'ICIR':<28} {icir_p5:>+16.3f} {icir_p6:>+13.3f}")
print(f"  {'Negative-IC folds':<28} {sum(1 for ic in p5_ics if ic<0):>16} {n_neg_p6:>13}")
print(f"  {'OOS Return':<28} {P_ML5['return']*100:>+15.1f}% {p6_return*100:>+12.1f}%")
print(f"  {'OOS Sharpe':<28} {P_ML5['sharpe']:>+16.3f} {p6_sharpe:>+13.3f}")
print(f"  {'Max Drawdown':<28} {P_ML5['maxdd']*100:>+15.1f}% {p6_maxdd*100:>+12.1f}%")
print()
print(f"LSTM IC  > LightGBM IC?      {'YES' if lstm_beats_lgbm_ic     else 'NO'}")
print(f"LSTM Sharpe > LightGBM Sharpe? {'YES' if lstm_beats_lgbm_sharpe else 'NO'}")
print(f"LSTM Sharpe > Buy & Hold?    {'YES' if lstm_beats_bah_sharpe  else 'NO'}")
print()

# Determine verdict
if lstm_beats_lgbm_sharpe and lstm_beats_lgbm_ic:
    verdict = (
        "HYPOTHESIS SUPPORTED: LSTM outperforms LightGBM on both IC and Sharpe. "
        "Multi-bar temporal dependencies add signal beyond single-bar features "
        "at the 1-day BTC horizon."
    )
elif lstm_beats_lgbm_sharpe or lstm_beats_lgbm_ic:
    verdict = (
        "HYPOTHESIS PARTIALLY SUPPORTED: LSTM improves on one metric but not both. "
        "Temporal memory provides some marginal benefit, but the improvement is not "
        "consistent across IC and equity."
    )
else:
    verdict = (
        "HYPOTHESIS REJECTED: LSTM does not outperform LightGBM at the 1-day horizon. "
        "Multi-bar temporal dependencies (30-bar window) do not add signal beyond "
        "single-bar features. Possible reasons: (1) at daily resolution, each bar "
        "already summarises the within-day sequence; (2) 30 bars may be too long — "
        "the signal is concentrated in 1–5 recent bars (matching LightGBM lag features); "
        "(3) limited training data (~300–500 sequences per fold) insufficient for LSTM."
    )

print(f"VERDICT: {verdict}")
print()
print("Next steps:")
print("  - If LSTM loses: try shorter seq_len (5–10 bars) targeting recent momentum")
print("  - Add attention layer to identify which historical bars drive the prediction")
print("  - Try 3–5d forward-return horizon where trend memory is more relevant")
print("  - Combine LSTM prediction with regime gate (P-ML3 Exp-C style gating)")

FINDING F13 — LSTM Forecaster (P-ML6) vs LightGBM (P-ML5)

Hypothesis: 30-bar LSTM captures multi-bar temporal dependencies that
single-bar LightGBM cannot, improving OOS IC and equity on 1d BTC.

Metric                           P-ML5 LightGBM    P-ML6 LSTM
--------------------------------------------------------------
  Mean OOS IC                           +0.0539       -0.0001
  ICIR                                   +0.888        -0.006
  Negative-IC folds                           1             2
  OOS Return                            +630.2%        -93.2%
  OOS Sharpe                             +0.927        -0.517
  Max Drawdown                           -68.0%        -94.7%

LSTM IC  > LightGBM IC?      NO
LSTM Sharpe > LightGBM Sharpe? NO
LSTM Sharpe > Buy & Hold?    NO

VERDICT: HYPOTHESIS REJECTED: LSTM does not outperform LightGBM at the 1-day horizon. Multi-bar temporal dependencies (30-bar window) do not add signal beyond single-bar features. Possible reasons: (1) at d